[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llama-certified/notebooks/day-08-ollama-rest-api.ipynb#scrollTo=a1b2c3d4)

---
# Day 8 · Serving with Ollama REST API and OpenAI-Compatible Endpoints
**certified-journeys / llama-certified** · Day 8 · Serving & Proxying

> **Goal for today:** Connect the OpenAI Python SDK to Ollama, build a FastAPI proxy with streaming, rate-limiting, and request logging, and benchmark it under concurrent load — all without changing a single line of your existing OpenAI client code.


In [ ]:
%pip install -q openai fastapi uvicorn httpx aiohttp requests starlette


## Step 1 · Ollama's OpenAI-Compatible Endpoint

Ollama exposes a drop-in OpenAI-compatible REST API at `http://localhost:11434/v1`.  
The only change needed in your existing OpenAI client code is two constructor arguments:

| OpenAI (cloud) | Ollama (local) |
|---|---|
| `base_url='https://api.openai.com/v1'` | `base_url='http://localhost:11434/v1'` |
| `api_key=os.environ['OPENAI_API_KEY']` | `api_key='ollama'` (any non-empty string) |
| Model: `'gpt-4o'` | Model: `'llama3.2:1b'` |

Every endpoint (`/chat/completions`, `/completions`, `/embeddings`, `/models`) works identically.

**In Colab** (no local Ollama server), we mock the HTTP layer so all code runs end-to-end.


In [ ]:
# ---------------------------------------------------------------------------
# Mock Ollama responses so this notebook runs in Colab without a local server.
# In production: remove the mock and point base_url at http://localhost:11434/v1
# ---------------------------------------------------------------------------
import json, time, random, unittest.mock as mock

MOCK_RESPONSE = {
    'id': 'chatcmpl-mock',
    'object': 'chat.completion',
    'created': int(time.time()),
    'model': 'llama3.2:1b',
    'choices': [{
        'index': 0,
        'message': {'role': 'assistant', 'content': 'Ollama is running locally and serving via its OpenAI-compatible REST API.'},
        'finish_reason': 'stop'
    }],
    'usage': {'prompt_tokens': 18, 'completion_tokens': 14, 'total_tokens': 32}
}

def _mock_post(url, **kwargs):
    """Return a fake response matching the OpenAI chat completions schema."""
    resp = mock.MagicMock()
    resp.status_code = 200
    resp.json.return_value = MOCK_RESPONSE
    resp.text = json.dumps(MOCK_RESPONSE)
    resp.headers = {'content-type': 'application/json'}
    return resp

print('Mock Ollama layer ready — swap base_url to http://localhost:11434/v1 on a real machine.')


**What just happened?**

- We defined a `MOCK_RESPONSE` dict that mirrors the exact JSON shape Ollama returns.
- `_mock_post` is a drop-in for `requests.post` — it returns the mock without hitting the network.
- **In production** the only change is `base_url='http://localhost:11434/v1'` and removing the mock.


## Step 2 · Connect the OpenAI Python SDK to Ollama

The OpenAI SDK accepts a `base_url` parameter — point it at Ollama and the entire API surface works unchanged.

```python
from openai import OpenAI
client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
response = client.chat.completions.create(
    model='llama3.2:1b',
    messages=[{'role': 'user', 'content': 'Hello from Ollama!'}]
)
```

The SDK sends an HTTP POST to `/v1/chat/completions` — identical to OpenAI's cloud endpoint.


In [ ]:
import requests, json

# Patch requests.post so our mock intercepts OpenAI SDK calls
with mock.patch('requests.Session.post', side_effect=_mock_post):
    from openai import OpenAI

    # Production: base_url='http://localhost:11434/v1'
    client = OpenAI(
        base_url='http://localhost:11434/v1',
        api_key='ollama'   # Ollama accepts any non-empty string
    )

    # Direct HTTP call to show what the SDK sends under the hood
    payload = {
        'model': 'llama3.2:1b',
        'messages': [{'role': 'user', 'content': 'What is Ollama?'}]
    }
    raw = _mock_post('http://localhost:11434/v1/chat/completions', json=payload)
    data = raw.json()

print('Model   :', data['model'])
print('Reply   :', data['choices'][0]['message']['content'])
print('Tokens  :', data['usage'])
print()
print('Endpoint: POST http://localhost:11434/v1/chat/completions')
print('Payload :', json.dumps(payload, indent=2))


**What just happened?**

- The OpenAI SDK serialises our `messages` list to a standard JSON payload and POSTs it.
- Ollama parses that payload, runs inference with the local model, and returns the same response schema as OpenAI.
- **Zero code changes** needed when switching between `gpt-4o` and `llama3.2:1b` — only `base_url` and `model` differ.


## Step 3 · List Available Models via the /models Endpoint

Ollama exposes `GET /v1/models` — the same endpoint OpenAI uses — so you can enumerate locally pulled models programmatically.

| Endpoint | Description |
|---|---|
| `GET /v1/models` | List all pulled models |
| `POST /v1/chat/completions` | Chat with a model |
| `POST /v1/completions` | Text completion (legacy) |
| `POST /v1/embeddings` | Generate embeddings |
| `DELETE /v1/models/{model}` | Remove a model |


In [ ]:
# Mock GET /v1/models response (production: requests.get('http://localhost:11434/v1/models'))
MOCK_MODELS = {
    'object': 'list',
    'data': [
        {'id': 'llama3.2:1b', 'object': 'model', 'created': 1700000000, 'owned_by': 'meta'},
        {'id': 'llama3.2:3b', 'object': 'model', 'created': 1700000100, 'owned_by': 'meta'},
        {'id': 'mistral:7b',  'object': 'model', 'created': 1700000200, 'owned_by': 'mistralai'},
    ]
}

print('Available models on this Ollama instance:')
print(f'{"Model ID":<25} {"Owner":<15} {"Created"}')
print('-' * 55)
for m in MOCK_MODELS['data']:
    print(f"{m['id']:<25} {m['owned_by']:<15} {m['created']}")

# Production code:
# import requests
# resp = requests.get('http://localhost:11434/v1/models')
# models = resp.json()['data']
# for m in models:
#     print(m['id'])


**What just happened?**

- `GET /v1/models` returns an `{object: 'list', data: [...]}` envelope — **identical** to OpenAI's response schema.
- Each entry has `id`, `object`, `created`, and `owned_by` fields.
- **Use case:** build model-selection dropdowns or health-check dashboards without hardcoding model names.


## Step 4 · Build a Streaming Chat FastAPI Proxy

A FastAPI proxy sits between your client and Ollama. Benefits:

- **Rate limiting** — prevent runaway requests from hammering the GPU
- **Logging** — capture every prompt/response for debugging and auditing
- **Auth** — add API-key checks before forwarding to the local server
- **Streaming** — SSE (Server-Sent Events) pass-through for real-time token delivery

```
Client  →  POST /v1/chat/completions  →  FastAPI proxy  →  Ollama :11434
Client  ←  SSE stream of tokens       ←  FastAPI proxy  ←  Ollama response
```


In [ ]:
# FastAPI streaming proxy — define the app (not started in Colab, run with uvicorn locally)
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import StreamingResponse
import httpx, time, json, asyncio
from collections import defaultdict

OLLAMA_BASE = 'http://localhost:11434'  # target server
RATE_LIMIT  = 5                         # max requests per 10-second window per client IP

app = FastAPI(title='Ollama Proxy')
_rate_buckets: dict = defaultdict(list)   # {ip: [timestamps]}
_request_log:  list = []                  # append-only in-memory log


def _check_rate_limit(client_ip: str) -> None:
    """Sliding-window rate limiter: max RATE_LIMIT requests per 10 s."""
    now = time.time()
    window = 10.0
    # Discard timestamps older than the window
    _rate_buckets[client_ip] = [
        t for t in _rate_buckets[client_ip] if now - t < window
    ]
    if len(_rate_buckets[client_ip]) >= RATE_LIMIT:
        raise HTTPException(status_code=429, detail='Rate limit exceeded — try again shortly')
    _rate_buckets[client_ip].append(now)


async def _stream_ollama(body: dict):
    """Async generator: forward the request to Ollama and yield SSE chunks."""
    body['stream'] = True
    async with httpx.AsyncClient(timeout=120.0) as ac:
        async with ac.stream('POST', f'{OLLAMA_BASE}/v1/chat/completions', json=body) as resp:
            async for chunk in resp.aiter_text():
                yield chunk  # each chunk is an SSE 'data: ...' line


@app.post('/v1/chat/completions')
async def chat_completions(request: Request):
    client_ip = request.client.host
    _check_rate_limit(client_ip)             # 429 if over limit

    body = await request.json()
    model = body.get('model', 'unknown')
    stream = body.get('stream', False)

    # Log the incoming request (non-blocking)
    _request_log.append({
        'ts': time.time(), 'ip': client_ip,
        'model': model, 'stream': stream,
        'n_messages': len(body.get('messages', []))
    })

    if stream:
        return StreamingResponse(
            _stream_ollama(body),
            media_type='text/event-stream'
        )

    # Non-streaming: forward and return full response
    async with httpx.AsyncClient(timeout=120.0) as ac:
        r = await ac.post(f'{OLLAMA_BASE}/v1/chat/completions', json=body)
    return r.json()


@app.get('/logs')
def get_logs():
    """Inspect recent request log."""
    return {'total': len(_request_log), 'recent': _request_log[-20:]}


print('FastAPI proxy defined.')
print('To run locally:  uvicorn day08_proxy:app --port 8080')
print('Then send:       curl http://localhost:8080/v1/chat/completions ...')
print(f'Rate limit:      {RATE_LIMIT} requests / 10 s per IP')


**What just happened?**

- `_check_rate_limit` uses a **sliding-window** approach — timestamps older than 10 s are discarded before counting.
- `_stream_ollama` is an **async generator** that pipes Ollama's SSE chunks straight to the client without buffering the entire response.
- `_request_log` is an in-process list — **in production** swap for a proper log sink (file, Redis, PostgreSQL).
- The `/logs` endpoint lets you inspect traffic without SSH-ing into the server.


## Step 5 · Simulate Streaming Token Delivery

When `stream=True`, Ollama sends **Server-Sent Events**: each line is `data: {JSON}`.  
The final line is always `data: [DONE]`.

```
data: {"choices":[{"delta":{"content":"Hello"},"finish_reason":null}]}
data: {"choices":[{"delta":{"content":" world"},"finish_reason":null}]}
data: [DONE]
```


In [ ]:
import time

# Simulate SSE streaming from Ollama
TOKENS = ['Ollama', ' serves', ' Llama', ' via', ' an', ' OpenAI-compatible', ' REST', ' API', '.']

def mock_sse_stream(tokens: list[str]):
    """Yield SSE lines as Ollama would send them."""
    for tok in tokens:
        chunk = {
            'id': 'chatcmpl-stream',
            'object': 'chat.completion.chunk',
            'choices': [{'index': 0, 'delta': {'content': tok}, 'finish_reason': None}]
        }
        yield f'data: {json.dumps(chunk)}\n\n'
        time.sleep(0.05)  # simulate ~20 tok/s
    yield 'data: [DONE]\n\n'


def consume_stream(stream_gen):
    """Parse the SSE stream and collect the full response text."""
    full_text = ''
    for line in stream_gen:
        line = line.strip()
        if not line.startswith('data:'):
            continue
        payload = line[len('data:'):].strip()
        if payload == '[DONE]':
            break
        chunk = json.loads(payload)
        tok = chunk['choices'][0]['delta'].get('content', '')
        full_text += tok
        print(tok, end='', flush=True)  # print tokens as they arrive
    return full_text


print('Streaming response:')
t0 = time.perf_counter()
full = consume_stream(mock_sse_stream(TOKENS))
elapsed = time.perf_counter() - t0
print(f'\n\nFull text : "{full}"')
print(f'Tokens    : {len(TOKENS)}')
print(f'Wall time : {elapsed:.2f}s  ({len(TOKENS)/elapsed:.1f} tok/s)')


**What just happened?**

- Each SSE chunk holds one `delta.content` token — identical to the OpenAI streaming format.
- `consume_stream` strips the `data:` prefix and parses JSON — the same logic your frontend would use.
- The `[DONE]` sentinel signals end-of-stream; always handle it or your loop hangs.
- **Real Ollama** adds `usage` to the final non-DONE chunk so you can track token counts without buffering.


## Step 6 · Rate Limiter Unit Tests

Verify the sliding-window rate limiter rejects requests over the limit but resets after the window expires.


In [ ]:
# Standalone rate-limiter (extracted from the FastAPI app for testing)
from collections import defaultdict
import time

class SlidingWindowRateLimiter:
    def __init__(self, max_calls: int, window_secs: float):
        self.max_calls = max_calls
        self.window    = window_secs
        self._buckets  = defaultdict(list)

    def is_allowed(self, client_id: str, now: float = None) -> bool:
        now = now or time.time()
        bucket = self._buckets[client_id]
        # Expire old timestamps
        self._buckets[client_id] = [t for t in bucket if now - t < self.window]
        if len(self._buckets[client_id]) >= self.max_calls:
            return False
        self._buckets[client_id].append(now)
        return True


# --- Tests ---
limiter = SlidingWindowRateLimiter(max_calls=3, window_secs=10.0)
t_base = 1000.0  # fixed 'now' so tests are deterministic

# First 3 calls: all allowed
results = [limiter.is_allowed('user1', now=t_base + i) for i in range(3)]
assert results == [True, True, True], f'Expected all True, got {results}'

# 4th call at t=1002: rejected (within same 10 s window)
assert not limiter.is_allowed('user1', now=t_base + 3), '4th call should be blocked'

# Call at t=1011: window has slid past first 3, so allowed again
assert limiter.is_allowed('user1', now=t_base + 11), 'Call after window should be allowed'

# Different client: independent bucket
assert limiter.is_allowed('user2', now=t_base), 'Different client should have its own bucket'

print('All rate-limiter tests passed.')
print(f'  max_calls = {limiter.max_calls}')
print(f'  window    = {limiter.window}s')


**What just happened?**

- We extracted `SlidingWindowRateLimiter` from the FastAPI app so it can be tested in isolation.
- **Sliding-window** means each new call evicts timestamps older than `window_secs` before counting — no burst allowed at window boundaries (unlike fixed-window limiters).
- Independent clients get their own bucket — one slow user doesn't penalise others.


## Step 7 · Request Logger and Audit Trail

Production proxies need structured logs for debugging, billing, and compliance.  
Key fields to capture per request:

| Field | Why |
|---|---|
| `ts` | Timestamp for latency analysis |
| `client_ip` | Rate-limit debugging and abuse tracking |
| `model` | Know which model is being used most |
| `prompt_tokens` | Estimate GPU load |
| `latency_ms` | SLO monitoring |
| `status` | Detect error spikes |


In [ ]:
import time, json, random
from dataclasses import dataclass, asdict, field
from typing import Optional

@dataclass
class RequestRecord:
    ts:            float
    client_ip:     str
    model:         str
    stream:        bool
    n_messages:    int
    prompt_tokens: int
    latency_ms:    float
    status:        int      # HTTP status code
    error:         Optional[str] = None


class RequestLogger:
    def __init__(self, max_records: int = 10_000):
        self._log: list[RequestRecord] = []
        self.max_records = max_records

    def record(self, rec: RequestRecord):
        self._log.append(rec)
        if len(self._log) > self.max_records:   # circular buffer
            self._log.pop(0)

    def recent(self, n: int = 10) -> list[dict]:
        return [asdict(r) for r in self._log[-n:]]

    def summary(self) -> dict:
        if not self._log:
            return {}
        latencies = [r.latency_ms for r in self._log]
        errors    = [r for r in self._log if r.status >= 400]
        return {
            'total_requests': len(self._log),
            'avg_latency_ms': round(sum(latencies) / len(latencies), 1),
            'p99_latency_ms': round(sorted(latencies)[int(len(latencies) * 0.99)], 1),
            'error_rate_pct': round(len(errors) / len(self._log) * 100, 1),
        }


# Simulate 50 logged requests
logger = RequestLogger()
for i in range(50):
    logger.record(RequestRecord(
        ts=time.time() - random.uniform(0, 300),
        client_ip=random.choice(['10.0.0.1', '10.0.0.2', '10.0.0.3']),
        model='llama3.2:1b',
        stream=random.choice([True, False]),
        n_messages=random.randint(1, 5),
        prompt_tokens=random.randint(20, 500),
        latency_ms=random.uniform(50, 3000),
        status=200 if random.random() > 0.05 else 429,
    ))

print('Request log summary:')
for k, v in logger.summary().items():
    print(f'  {k:<22}: {v}')

print('\nMost recent request:')
print(json.dumps(logger.recent(1)[0], indent=2))


**What just happened?**

- `RequestRecord` is a typed dataclass — `asdict()` serialises it to JSON for persistence.
- The logger keeps a **circular buffer** (max 10 000 records) so memory usage is bounded.
- `summary()` computes p99 latency by sorting — for production use `numpy.percentile` or a streaming histogram (t-digest).
- **Production swap:** replace the in-process list with writes to PostgreSQL / BigQuery / CloudWatch.


## Step 8 · Benchmark Concurrent Requests

Compare response time under 5 simultaneous users hitting the proxy vs. calling Ollama directly.

We simulate both with `asyncio` and a mocked HTTP layer so the test runs in Colab without a GPU.


In [ ]:
import asyncio, time, random, statistics

# Simulate a single inference call with a realistic latency distribution
async def mock_ollama_call(request_id: int, overhead_ms: float = 0.0) -> dict:
    """Simulate one POST /v1/chat/completions with random latency."""
    # Realistic: ~200-800 ms for a short completion on llama3.2:1b on CPU
    latency = random.uniform(0.20, 0.80) + overhead_ms / 1000
    await asyncio.sleep(latency)
    return {'id': request_id, 'latency_ms': round(latency * 1000, 1), 'tokens': random.randint(20, 80)}


async def run_concurrent(n_users: int, overhead_ms: float = 0.0) -> dict:
    """Fire n_users simultaneous requests and collect timing stats."""
    t0 = time.perf_counter()
    results = await asyncio.gather(*[
        mock_ollama_call(i, overhead_ms=overhead_ms) for i in range(n_users)
    ])
    wall_time = (time.perf_counter() - t0) * 1000  # ms
    latencies = [r['latency_ms'] for r in results]
    total_tokens = sum(r['tokens'] for r in results)
    return {
        'n_users':      n_users,
        'wall_ms':      round(wall_time, 1),
        'avg_ms':       round(statistics.mean(latencies), 1),
        'p99_ms':       round(sorted(latencies)[int(len(latencies) * 0.99)], 1),
        'total_tokens': total_tokens,
        'tok_per_sec':  round(total_tokens / (wall_time / 1000), 1),
    }


# Direct Ollama (no proxy overhead)
direct = await run_concurrent(n_users=5, overhead_ms=0)

# Via FastAPI proxy (adds ~5–20 ms per request for routing + logging)
proxy  = await run_concurrent(n_users=5, overhead_ms=12)  # 12 ms proxy overhead

print('Concurrent load benchmark (5 simultaneous users):')
print(f'{"Metric":<20} {"Direct":>12} {"Via Proxy":>12} {"Delta":>10}')
print('-' * 55)
for key in ('wall_ms', 'avg_ms', 'p99_ms', 'tok_per_sec'):
    d, p = direct[key], proxy[key]
    delta = f'+{p-d:.1f}' if p >= d else f'{p-d:.1f}'
    print(f'{key:<20} {d:>12} {p:>12} {delta:>10}')


**What just happened?**

- `asyncio.gather` fires all 5 requests at once, so wall time is dominated by the **slowest single request**, not the sum of all latencies.
- The proxy adds ~12 ms overhead per request — acceptable for most applications.
- **Key insight:** Ollama is single-threaded by default. Under 5 concurrent users, requests queue internally — the bottleneck is GPU, not the proxy.
- In production, set `OLLAMA_NUM_PARALLEL=4` to allow true concurrency at the cost of VRAM.


In [ ]:
# Challenge: Build a simple load-tester that fires N concurrent requests and
# reports: avg latency, p95 latency, total throughput (tokens/sec), and error rate.
#
# Requirements:
#   1. Accept parameters: n_users (int), model (str), prompt (str)
#   2. Send real HTTP requests to http://localhost:11434/v1/chat/completions
#      (use mock_ollama_call for Colab; swap in real httpx calls locally)
#   3. Handle 429 rate-limit errors: count them separately, don't let them crash the test
#   4. Print a formatted summary table at the end
#
# Scaffold:
async def load_test(n_users: int, model: str = 'llama3.2:1b', prompt: str = 'Say hello.'):
    # TODO: fire n_users concurrent requests
    # TODO: collect latencies, token counts, and errors
    # TODO: compute avg, p95, tok/s, error_rate
    # TODO: print summary table
    pass

# await load_test(n_users=10)
print('Implement load_test() above, then call: await load_test(n_users=10)')


---
## Day 8 key concepts recap

| Concept | What to remember |
|---|---|
| OpenAI compatibility | Just set `base_url='http://localhost:11434/v1'` and `api_key='ollama'` — zero other changes |
| SSE streaming | Each chunk is `data: {JSON}\n\n`; the final chunk is `data: [DONE]\n\n` |
| Sliding-window rate limiter | Evict old timestamps before counting — prevents bursts at window boundaries |
| Proxy overhead | ~5–20 ms per request; measure p99, not avg, to detect tail latency |
| Concurrent users | Ollama queues requests internally; set `OLLAMA_NUM_PARALLEL` for true GPU concurrency |

> **Tip:** Ollama's OpenAI-compatible endpoint is at `http://localhost:11434/v1`. Just set `base_url` to that URL in the OpenAI client — zero code changes needed to switch between OpenAI and local Llama.

---
## What's next
**Day 9** → Benchmarking — measure prefill latency, decode tokens/sec, and peak RAM across model variants; build an LLM judge to score quality on 20 held-out prompts.

Mark Day 8 complete in your [tracker](../index.html).
